# Questão 2 - Schema Datasets [CSV to PostgreSQL]

In [9]:
#Regras:
#* Considere todos os CSV como arquivos de fonte.
#* Utilize obrigatoriamente Python 3.
#* Utilize somente bibliotecas padrão do Python 3 (csv, os, datetime e etc.) e python puro. Soluções que utilizarem bibliotecas como pandas, dask, polars serão desconsideradas.
#* Considere o banco de destino como sendo um PostgreSQL.

In [10]:
#Crie um script python que possa ler os CSV de um diretório 
#e gere um único arquivo de saída (schema.sql) com as instruções de criação de uma tabela para cada arquivo CSV 
#usando somente bibliotecas de acordo com as instruções anteriores.

In [11]:
#Importando as bibliotecas necessárias
import os
import csv
from datetime import datetime

In [12]:
#Definindo o diretório onde os arquivos CSVs estão localizados e o nome do arquivo de saída
DATASET_DIR = "Dataset"
OUTPUT_FILE = "schema.sql"

In [13]:
#Função para identificar colunas que são identificadores (id)
def id_type_column(column_name):
    """ Verifica se a coluna representa um identificador (id) para se tratado separadamente. """
    column_name = column_name.strip().lower()
    return (
        column_name == "id"
        or column_name.endswith("_id")
    )

#Função para inferir o tipo de dados de cada coluna com base nos valores encontrados nos CSVs
def infer_type(column_name, values):
    """ Infere o tipo de SQL de acordo com o que o PostgreSQL aceita no schema com base nos valores encontrados. """
    values = [value.strip() for value in values if value.strip()]

    #Colunas de IDs serão sempre tratados como TEXT
    if id_type_column(column_name):
        return "TEXT"

    values = [
        value.strip()
        for value in values
        if value.strip() != ""
    ]

    #TEXT
    if not values:
        return "TEXT"

    #INTEGER
    try:
        for value in values:
            int(value)
        return "INTEGER"
    except ValueError:
        pass

    #NUMERIC
    try:
        for value in values:
            float(value)
        return "NUMERIC"
    except ValueError:
        pass

    #DATE/TIMESTAMP
    date_formats = [
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S"
    ]
    for date_format in date_formats:
        try:
            for value in values:
                datetime.strptime(value, date_format)
            if "H" in date_format:
                return "TIMESTAMP"
            return "DATE"
        except ValueError:
            continue
    return "TEXT"

In [ ]:
#Funções para tratar nomes de tabelas e colunas para o PostgreSQL
def created_table_name(filename):
    """ Utiliza o nome do arquivo CSV como nome da tabela SQL."""
    table_name = os.path.splitext(filename)[0]

    return table_name.lower()

def created_column_name(column):
    """ Normaliza o nome das colunas para PostgreSQL."""

    #tratamento de possiveis espaços e caracteres especiais no nome da coluna
    column = column.strip()
    column = column.replace(" ", "_")
    column = column.replace("-", "_")

    return column.lower()

In [15]:
#Função principal para gerar o schema SQL a partir dos arquivos CSVs
def generated_schema():
    """ Gera o schema SQL a partir dos arquivos CSVs no diretório especificado. """
    sql_statements = []
    files = sorted(os.listdir(DATASET_DIR))

    for filename in files:
        #Apenas selecionar arquivos .csv no diretório especificado
        if not filename.lower().endswith(".csv"):
            continue

        filepath = os.path.join(DATASET_DIR, filename)
        table_name = created_table_name(filename)
    
        print(f"Processando: {filename}")

        with open(filepath, "r", encoding="utf-8-sig", newline="") as csvfile:

            reader = csv.reader(csvfile)
            header = next(reader)
            rows = list(reader)
            columns = list(zip(*rows)) if rows else [[] for _ in header]
            column_definitions = []

            for column_name, values in zip(header, columns):
                original_column_name = column_name #Para verifcar as colunas Ids
                column_name = created_column_name(column_name)
                data_type = infer_type(original_column_name, values)
                column_definitions.append(
                    f'    "{column_name}" {data_type}'
                )

            create_table = (
                f'CREATE TABLE "{table_name}" (\n'
                + ",\n".join(column_definitions)
                + "\n);\n"
            )

            sql_statements.append(create_table)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as sqlfile:
        sqlfile.write(
            "-- Schema gerado automaticamente LH Nautical\n\n"
        )
        sqlfile.write(
            "\n".join(sql_statements)
        )


In [16]:
#Chamando função principal para gerar o schema SQL a partir dos arquivos CSVs
generated_schema()

Processando: addresses.csv
Processando: attributes.csv
Processando: brands.csv
Processando: categories.csv
Processando: customers.csv
Processando: employees.csv
Processando: fiscal_invoices.csv
Processando: goods_receipt_items.csv
Processando: goods_receipts.csv
Processando: locations.csv
Processando: order_items.csv
Processando: orders.csv
Processando: payments.csv
Processando: product_suppliers.csv
Processando: product_variants.csv
Processando: products.csv
Processando: purchase_order_items.csv
Processando: purchase_orders.csv
Processando: return_items.csv
Processando: returns.csv
Processando: stock_levels.csv
Processando: stock_movements.csv
Processando: suppliers.csv
Processando: variant_attribute_values.csv
